In [ ]:
# start coding himport xarray as xr
from pyclim_noresm.aerosol_forcing import calc_direct_aerosol_radiative_effect
import copy
import time
import xarray as xr
import logging, traceback
logging.basicConfig(filename=snakemake.log[0],
                    level=logging.INFO,
                    format='%(asctime)s %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    )



In [ ]:
VARS = snakemake.config['variables']

ERFt = xr.open_dataset(snakemake.input.ERFt)
ERFtaf = xr.open_dataset(snakemake.input.ERFtaf)

directEffect = calc_direct_aerosol_radiative_effect(ERFt[VARS[snakemake.wildcards.vName][0]],
                                    ERFtaf[VARS[snakemake.wildcards.vName][1]])

directEffect = directEffect.to_dataset(name=snakemake.wildcards.vName)
directEffect.attrs=copy.copy(ERFt.attrs)
directEffect = directEffect.assign_attrs(variable_id=snakemake.wildcards.vName)
directEffect.attrs['title']='Aerosol direct radiative effect'
directEffect.attrs['history'] = f'@{time.ctime()} Generated by: {snakemake.rule} ' + directEffect.attrs['history']
directEffect.attrs['source'] = ', '.join(snakemake.input) + ', ' + directEffect.attrs['source']
directEffect.to_netcdf(snakemake.output.outpath)